# Multi-Omic Data Preparation and Quality Control

This notebook handles the initial data loading, quality control, and preprocessing of multi-omic datasets for heart disease stratification. 

We integrate data from multiple omic layers: genomics, transcriptomics, proteomics, and metabolomics. Each layer requires specific quality control and normalization approaches to ensure data quality and comparability across samples.

## Setup and Dependencies

Import all necessary libraries for data processing, visualization, and statistical analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import pdist, squareform
import warnings

warnings.filterwarnings('ignore')

np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('All dependencies loaded successfully')

## 1. Data Loading and Preprocessing

Load multi-omic datasets from GTEx or similar sources. Each omic layer has different characteristics and requires tailored loading and preprocessing approaches.

In [ ]:
def load_omic_data(filepath, omic_type):
    """
    Load omic data from csv file with basic validation
    
    Args:
        filepath: path to data file
        omic_type: type of omic (genomics, transcriptomics, proteomics, metabolomics)
    
    Returns:
        DataFrame with data loaded
    """
    try:
        data = pd.read_csv(filepath, index_col=0)
        print(f'{omic_type}: Loaded {data.shape[0]} features from {data.shape[1]} samples')
        return data
    except Exception as e:
        print(f'Error loading {omic_type}: {str(e)}')
        return None


def create_sample_data():
    """
    Create sample multi-omic data for demonstration purposes
    """
    n_samples = 50
    sample_ids = [f'patient_{i:03d}' for i in range(1, n_samples + 1)]
    
    genomics = pd.DataFrame(
        np.random.choice([0, 1, 2], size=(500, n_samples)),
        columns=sample_ids,
        index=[f'SNP_{i}' for i in range(500)]
    )
    
    transcriptomics = pd.DataFrame(
        np.random.exponential(scale=2, size=(5000, n_samples)),
        columns=sample_ids,
        index=[f'GENE_{i}' for i in range(5000)]
    )
    
    proteomics = pd.DataFrame(
        np.random.normal(loc=5, scale=1, size=(1500, n_samples)),
        columns=sample_ids,
        index=[f'PROTEIN_{i}' for i in range(1500)]
    )
    
    metabolomics = pd.DataFrame(
        np.random.lognormal(mean=0, sigma=1, size=(300, n_samples)),
        columns=sample_ids,
        index=[f'METABOLITE_{i}' for i in range(300)]
    )
    
    clinical = pd.DataFrame({
        'patient_id': sample_ids,
        'age': np.random.randint(40, 80, n_samples),
        'sex': np.random.choice(['M', 'F'], n_samples),
        'ejection_fraction': np.random.uniform(20, 70, n_samples),
        'bnp': np.random.lognormal(mean=5, sigma=1, size=n_samples),
        'nyha_class': np.random.choice([1, 2, 3, 4], n_samples),
        'hf_status': np.random.choice(['control', 'HF'], n_samples),
        'outcome': np.random.choice([0, 1], n_samples)
    })
    
    clinical.set_index('patient_id', inplace=True)
    
    return genomics, transcriptomics, proteomics, metabolomics, clinical


print('Loading sample multi-omic data for demonstration...')
genomics, transcriptomics, proteomics, metabolomics, clinical = create_sample_data()

print(f'Genomics shape: {genomics.shape}')
print(f'Transcriptomics shape: {transcriptomics.shape}')
print(f'Proteomics shape: {proteomics.shape}')
print(f'Metabolomics shape: {metabolomics.shape}')
print(f'Clinical data shape: {clinical.shape}')

## 2. Exploratory Data Analysis of Multi-Omic Data

Visualize the basic characteristics of each omic layer to understand data quality, sample relationships, and potential batch effects.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(genomics.values.flatten(), bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Genomics Distribution (Allele Counts)')
axes[0, 0].set_xlabel('Allele Count')
axes[0, 0].set_ylabel('Frequency')

axes[0, 1].hist(transcriptomics.values.flatten(), bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Transcriptomics Distribution (Expression)')
axes[0, 1].set_xlabel('Expression Level')
axes[0, 1].set_ylabel('Frequency')

axes[1, 0].hist(proteomics.values.flatten(), bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Proteomics Distribution (Abundance)')
axes[1, 0].set_xlabel('Protein Abundance')
axes[1, 0].set_ylabel('Frequency')

axes[1, 1].hist(metabolomics.values.flatten(), bins=50, edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Metabolomics Distribution (Concentration)')
axes[1, 1].set_xlabel('Metabolite Concentration')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../results/qc_reports/01_distribution_plots.png', dpi=300, bbox_inches='tight')
plt.show()

print('Distribution plots saved')

print('\nBasic statistics for each omic layer:')
print('\nGenomics (SNPs):')
print(f'  Mean: {genomics.values.mean():.3f}, Std: {genomics.values.std():.3f}')
print(f'  Min: {genomics.values.min()}, Max: {genomics.values.max()}')

print('\nTranscriptomics (Gene Expression):')
print(f'  Mean: {transcriptomics.values.mean():.3f}, Std: {transcriptomics.values.std():.3f}')
print(f'  Min: {transcriptomics.values.min():.3f}, Max: {transcriptomics.values.max():.3f}')

print('\nProteomics (Protein Abundance):')
print(f'  Mean: {proteomics.values.mean():.3f}, Std: {proteomics.values.std():.3f}')
print(f'  Min: {proteomics.values.min():.3f}, Max: {proteomics.values.max():.3f}')

print('\nMetabolomics (Metabolite Concentration):')
print(f'  Mean: {metabolomics.values.mean():.3f}, Std: {metabolomics.values.std():.3f}')
print(f'  Min: {metabolomics.values.min():.3f}, Max: {metabolomics.values.max():.3f}')

## 3. Feature Selection Across Omics Layers

Apply statistical filtering to reduce dimensionality while preserving biologically relevant features. We use variance-based filtering and correlation analysis to identify the most informative features.

In [ ]:
def select_features_by_variance(data, percentile=75):
    """
    Select features by variance. Keep features with variance in top percentile.
    
    Args:
        data: feature matrix (features x samples)
        percentile: threshold percentile for variance
    
    Returns:
        filtered data and indices of selected features
    """
    variances = np.var(data.values, axis=1)
    threshold = np.percentile(variances, percentile)
    selected = variances >= threshold
    
    print(f'Variance filtering: {data.shape[0]} -> {np.sum(selected)} features')
    return data.loc[selected], np.where(selected)[0]


def normalize_data(data, method='zscore'):
    """
    Normalize data to zero mean and unit variance
    
    Args:
        data: feature matrix
        method: normalization method (zscore or minmax)
    
    Returns:
        normalized data
    """
    if method == 'zscore':
        mean = data.values.mean(axis=1, keepdims=True)
        std = data.values.std(axis=1, keepdims=True)
        normalized = (data.values - mean) / (std + 1e-8)
    elif method == 'minmax':
        min_val = data.values.min(axis=1, keepdims=True)
        max_val = data.values.max(axis=1, keepdims=True)
        normalized = (data.values - min_val) / (max_val - min_val + 1e-8)
    
    return pd.DataFrame(normalized, index=data.index, columns=data.columns)


print('Applying feature selection by variance filtering...\n')

genomics_filt, _ = select_features_by_variance(genomics, percentile=75)
transcriptomics_filt, _ = select_features_by_variance(transcriptomics, percentile=75)
proteomics_filt, _ = select_features_by_variance(proteomics, percentile=75)
metabolomics_filt, _ = select_features_by_variance(metabolomics, percentile=75)

print('\nNormalizing selected features...\n')

genomics_norm = normalize_data(genomics_filt, method='zscore')
transcriptomics_norm = normalize_data(transcriptomics_filt, method='zscore')
proteomics_norm = normalize_data(proteomics_filt, method='zscore')
metabolomics_norm = normalize_data(metabolomics_filt, method='zscore')

print('Feature selection and normalization complete')
print(f'\nFinal feature counts:')
print(f'  Genomics: {genomics_norm.shape[0]} variants')
print(f'  Transcriptomics: {transcriptomics_norm.shape[0]} genes')
print(f'  Proteomics: {proteomics_norm.shape[0]} proteins')
print(f'  Metabolomics: {metabolomics_norm.shape[0]} metabolites')

## 4. Pathway Enrichment Analysis

Map significant features to known heart disease related pathways. This step connects individual molecular measurements to biological processes implicated in cardiovascular disease.

In [ ]:
cardiac_pathways = {
    'Mitochondrial Function': [
        'NDUFA1', 'NDUFA2', 'NDUFB3', 'SDHA', 'SDHB', 'CYC1', 'ATP5A1', 'ATP5B',
        'COX5A', 'COX5B', 'COX6A1', 'COX7A1', 'COX8A', 'UQCRC1', 'UQCRC2'
    ],
    'Inflammatory Signaling': [
        'TNF', 'IL1A', 'IL1B', 'IL6', 'IL8', 'TNFRSF1A', 'TNFRSF1B', 'IL6R',
        'NFKB1', 'NFKB2', 'RELA', 'RELB', 'IKBKG', 'NFKBIA', 'NFKBIZ'
    ],
    'Cardiac Contraction': [
        'MYH7', 'MYL2', 'MYL3', 'TNNT2', 'TNNC1', 'ACTC1', 'TPM1', 'MYBPC3',
        'ACTN2', 'Z DISC1', 'LMNA', 'TMEM43'
    ],
    'Fibrosis Pathways': [
        'TGFB1', 'TGFB2', 'TGFB3', 'SMAD2', 'SMAD3', 'SMAD4', 'SMAD7',
        'COL1A1', 'COL1A2', 'COL3A1', 'COL5A1', 'FN1'
    ],
    'Energy Metabolism': [
        'PRKAA1', 'PRKAA2', 'AMPK', 'MTOR', 'ULK1', 'PFKFB2', 'PFKFB3',
        'PHPKB', 'GAPDH', 'HK1', 'HK2'
    ]
}

print('Cardiac pathways for analysis:')
print('=' * 50)
for pathway, genes in cardiac_pathways.items():
    print(f'{pathway}: {len(genes)} genes')

print('\nNote: In a real analysis, pathway enrichment would use statistical')
print('tests like GSEA (Gene Set Enrichment Analysis) to determine if')
print('features are significantly upregulated or downregulated. This')
print('requires comparison between disease and control samples.')

## 5. Machine Learning Model Development

Build classification models to stratify patients into disease subtypes. We combine features from all omic layers to train ensemble methods that capture multi-level relationships between genomic, transcriptomic, proteomic, and metabolomic alterations.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

def combine_omics(genomics_df, transcriptomics_df, proteomics_df, metabolomics_df):
    """
    Combine all omic layers into single feature matrix
    """
    combined_list = []
    feature_names = []
    
    for df, prefix in [
        (genomics_df, 'gen'),
        (transcriptomics_df, 'trx'),
        (proteomics_df, 'prot'),
        (metabolomics_df, 'met')
    ]:
        combined_list.append(df.T.values)
        feature_names.extend([f'{prefix}_{i}' for i in range(df.shape[0])])
    
    combined = np.hstack(combined_list)
    return combined, feature_names


print('Combining all omic layers...')
X_combined, feature_names = combine_omics(
    genomics_norm, 
    transcriptomics_norm, 
    proteomics_norm, 
    metabolomics_norm
)

print(f'Combined feature matrix: {X_combined.shape[0]} samples, {X_combined.shape[1]} features')

y = clinical['hf_status'].map({'control': 0, 'HF': 1}).values

X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'\nTraining set: {X_train_scaled.shape}')
print(f'Test set: {X_test_scaled.shape}')
print(f'Class distribution (train): {np.bincount(y_train)}')
print(f'Class distribution (test): {np.bincount(y_test)}')

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

print('\nTraining random forest model...')
rf_model.fit(X_train_scaled, y_train)

y_pred = rf_model.predict(X_test_scaled)
y_pred_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

print('\n' + '=' * 50)
print('MODEL PERFORMANCE')
print('=' * 50)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall: {recall_score(y_test, y_pred):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}')

print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Control', 'HF']))

## 6. Genetic Pathway Stratification

Develop a stratification system that groups patients into molecular subtypes based on which pathways are predominantly dysregulated in their samples.

In [ ]:
from sklearn.cluster import KMeans

def create_pathway_scores(transcriptomics_df, pathway_dict):
    """
    Calculate pathway activity scores for each sample
    Simple approach: average expression of pathway genes
    """
    pathway_scores = {}
    
    for pathway_name, genes in pathway_dict.items():
        available_genes = [g for g in genes if g in transcriptomics_df.index]
        
        if len(available_genes) > 0:
            pathway_expr = transcriptomics_df.loc[available_genes]
            pathway_scores[pathway_name] = pathway_expr.mean(axis=0).values
        else:
            print(f'Warning: No genes found for pathway {pathway_name}')
    
    scores_df = pd.DataFrame(pathway_scores).T
    return scores_df


pathway_scores = create_pathway_scores(transcriptomics_norm, cardiac_pathways)
print(f'Pathway scores matrix shape: {pathway_scores.shape}')
print('\nPathway scores (first 5 samples):')
print(pathway_scores.iloc[:, :5])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
subtypes = kmeans.fit_predict(pathway_scores.T)

print(f'\nClustering results:')
print(f'  Number of subtypes identified: {len(np.unique(subtypes))}')
print(f'  Subtype distribution:')
for subtype in np.unique(subtypes):
    count = np.sum(subtypes == subtype)
    pct = 100 * count / len(subtypes)
    print(f'    Subtype {subtype}: {count} patients ({pct:.1f}%)')

subtype_df = pd.DataFrame({
    'patient_id': pathway_scores.columns,
    'subtype': subtypes.astype(str)
})
subtype_df.set_index('patient_id', inplace=True)

print('\nSubtype assignments (first 10 patients):')
print(subtype_df.head(10))

## 7. Treatment Recommendation Engine

Map dysregulated pathways to evidence-based therapeutic interventions. Each subtype receives ranked recommendations based on the molecular mechanisms identified in their disease profile.

In [ ]:
subtype_drugs = {
    0: {
        'Mitochondrial Function': ['CoQ10', 'Ubiquinol', 'L-carnitine', 'Riboflavin'],
        'Energy Metabolism': ['SGLT2 inhibitors', 'Beta-blockers', 'ARNi'],
        'Confidence': 0.85
    },
    1: {
        'Inflammatory Signaling': ['IL-6 antagonist (Anakinra)', 'TNF-alpha inhibitors', 'Colchicine'],
        'Immune modulation': ['Immunosuppression', 'Corticosteroids (short-term)'],
        'Confidence': 0.78
    },
    2: {
        'Fibrosis Pathways': ['SGLT2i', 'Finerenone', 'Pirfenidone'],
        'TGF-beta inhibition': ['Pentraxin-2 inhibitors', 'Abatacept'],
        'Confidence': 0.82
    }
}

def generate_treatment_recommendations(subtype, pathway_scores_row, subtype_drugs):
    """
    Generate personalized treatment recommendations based on subtype
    and pathway activation patterns
    """
    if subtype not in subtype_drugs:
        return None
    
    recommendations = {
        'subtype': subtype,
        'treatments': []
    }
    
    for pathway, drugs in subtype_drugs[subtype].items():
        if pathway == 'Confidence':
            recommendations['confidence'] = drugs
            continue
        
        recommendations['treatments'].append({
            'pathway': pathway,
            'recommended_drugs': drugs,
            'mechanism': f'Targets {pathway} dysregulation'
        })
    
    return recommendations


print('TREATMENT RECOMMENDATIONS BY SUBTYPE')
print('=' * 70)

for subtype in np.unique(subtypes):
    n_patients = np.sum(subtypes == subtype)
    recommendations = generate_treatment_recommendations(subtype, None, subtype_drugs)
    
    print(f'\nSubtype {subtype} ({n_patients} patients, {100*n_patients/len(subtypes):.1f}%)')
    print('-' * 70)
    print(f'Confidence: {recommendations["confidence"]:.0%}')
    
    for treatment in recommendations['treatments']:
        print(f'\n  {treatment["pathway"]}:')
        for drug in treatment['recommended_drugs']:
            print(f'    - {drug}')

print('\n' + '=' * 70)
print('Note: These are demonstration recommendations. Real recommendations')
print('would be based on clinical trial data, mechanistic validation,')
print('and individual patient factors.')

## 8. Model Validation and Performance Metrics

Evaluate model robustness and generalizability using multiple validation approaches. Generate comprehensive performance metrics to assess stratification quality and predictive accuracy.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_curve, auc

cv_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring='roc_auc')

print('COMPREHENSIVE MODEL VALIDATION')
print('=' * 70)
print('\nCross-Validation Results (5-Fold):')
print(f'  ROC-AUC scores: {cv_scores}')
print(f'  Mean CV-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

print('\nTest Set Performance:')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'  Precision: {precision_score(y_test, y_pred):.4f}')
print(f'  Recall:    {recall_score(y_test, y_pred):.4f}')
print(f'  F1-Score:  {f1_score(y_test, y_pred):.4f}')
print(f'  ROC-AUC:   {roc_auc_score(y_test, y_pred_proba):.4f}')

fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[0].diagonal_line = axes[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Chance')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve - Multi-Omic Model')
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.3)

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1], cbar=False)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix')
axes[1].set_xticklabels(['Control', 'HF'])
axes[1].set_yticklabels(['Control', 'HF'])

plt.tight_layout()
plt.savefig('../results/qc_reports/02_model_performance.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nTop 15 important features:')
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance.head(15).to_string())

print('\nProcessing and preparation complete!')
print('Data ready for multi-omic integration analysis.')